<a href="https://colab.research.google.com/github/chaithu-gif/AI_LAB/blob/main/week7_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
def is_variable(term):
    """Returns True if the term is a variable (prefixed with '?')."""
    return isinstance(term, str) and term.startswith('?')


def occurs_check(var, x, theta):
    """Prevents cyclic dependencies by checking if 'var' occurs inside term 'x'."""
    if var == x:
        return True
    elif is_variable(x) and x in theta:
        return occurs_check(var, theta[x], theta)
    elif isinstance(x, (tuple, list)):
        return any(occurs_check(var, arg, theta) for arg in x[1:])
    return False


def unify_var(var, x, theta):
    """Unifies a variable with an expression x under existing bindings theta."""
    if var in theta:
        return unify(theta[var], x, theta)
    elif is_variable(x) and x in theta:
        return unify(var, theta[x], theta)
    elif occurs_check(var, x, theta):
        return None  # Occurs check failed: prevents cyclic infinite trees
    else:
        new_theta = theta.copy()
        new_theta[var] = x
        return new_theta


def unify(x, y, theta=None):
    """
    Computes the Most General Unifier (MGU) for expressions x and y.
    Returns a substitution dictionary if unifiable, or None if unification fails.
    """
    if theta is None:
        theta = {}

    if x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, (tuple, list)) and isinstance(y, (tuple, list)):
        if len(x) != len(y):
            return None
        # Unify heads (predicates/functions), then pass substitution to unify tails (arguments)
        first_unification = unify(x[0], y[0], theta)
        if first_unification is None:
            return None
        return unify(x[1:], y[1:], first_unification)
    else:
        return None


def apply_substitution(expr, theta):
    """Applies a substitution dictionary theta recursively to an expression."""
    if theta is None:
        return None
    if is_variable(expr):
        if expr in theta:
            return apply_substitution(theta[expr], theta)
        return expr
    elif isinstance(expr, (tuple, list)):
        return type(expr)(apply_substitution(arg, theta) for arg in expr)
    else:
        return expr

  # Helper to run and display unification tests
def test_unify(expr1, expr2):
    mgu = unify(expr1, expr2)
    print(f"Expr 1: {expr1}")
    print(f"Expr 2: {expr2}")
    print(f"MGU:    {mgu}")
    if mgu is not None:
        print(f"Result: {apply_substitution(expr1, mgu)}\n")
    else:
        print("Result: Unification Failed\n")


# 1. Basic Variable Matching: Knows(John, ?x) and Knows(?y, Jane)
test_unify(('Knows', 'John', '?x'), ('Knows', '?y', 'Jane'))

# 2. Nested Functions: f(?x, g(?x)) and f(a, ?y)
test_unify(('f', '?x', ('g', '?x')), ('f', 'a', '?y'))

# 3. Occurs Check Failure: ?x and f(?x)
test_unify('?x', ('f', '?x'))

# 4. Predicate Mismatch: P(?x) and Q(?x)
test_unify(('P', '?x'), ('Q', '?x'))

Expr 1: ('Knows', 'John', '?x')
Expr 2: ('Knows', '?y', 'Jane')
MGU:    {'?y': 'John', '?x': 'Jane'}
Result: ('Knows', 'John', 'Jane')

Expr 1: ('f', '?x', ('g', '?x'))
Expr 2: ('f', 'a', '?y')
MGU:    {'?x': 'a', '?y': ('g', '?x')}
Result: ('f', 'a', ('g', 'a'))

Expr 1: ?x
Expr 2: ('f', '?x')
MGU:    None
Result: Unification Failed

Expr 1: ('P', '?x')
Expr 2: ('Q', '?x')
MGU:    None
Result: Unification Failed

